In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

backend = BasicSimulator()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 59.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 11.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=9e008af7c4b90d23745abd709405850bca60b651dc93ac206a9d0bd10b99d0c5
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.

## Overview

Sequential simulation of BB84 between Alice and Bob, no eavesdropper. Sections are labelled by agent ([ALICE], [BOB]) and channel ([CLASSICAL]) so the boundaries are easy to follow.

Random choices come from quantum measurements of $H|0\rangle$ — Python's `random` is never used.

Protocol steps:
1. Alice picks random bits and bases
2. Bob picks random measurement bases
3. Alice encodes, Bob measures (the quantum channel)
4. Sifting — keep positions where bases agree
5. Error check — reveal a sample, abort if error rate exceeds threshold

## Quantum random bit generator

Prepare $|+\rangle = H|0\rangle$ and measure in Z. Outcome is 0 or 1, each with probability $\tfrac{1}{2}$.

Used by Alice and Bob for every random choice they need to make.

In [3]:
def quantum_random_bits(n: int) -> list[int]:
    """Return n random bits, each from measuring an independent |+> qubit."""
    out = []
    for _ in range(n):
        qc = QuantumCircuit(1, 1)
        qc.h(0)
        qc.measure(0, 0)
        tqc = transpile(qc, backend)
        counts = backend.run(tqc, shots=1).result().get_counts()
        out.append(int(next(iter(counts))))
    return out

# Sanity check: 200 bits should be roughly balanced.
check = quantum_random_bits(200)
print(f"[RNG] 200 bits: {sum(check)} ones, {200 - sum(check)} zeros")
print(f"[RNG] first 20: {check[:20]}")

[RNG] 200 bits: 102 ones, 98 zeros
[RNG] first 20: [1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0]


## Alice — encoding

Two bases, two bit values, four possible states:

| Bit | Z basis (0) | X basis (1) |
|-----|-------------|-------------|
| 0   | $\lvert 0\rangle$ | $\lvert +\rangle$ |
| 1   | $\lvert 1\rangle$ | $\lvert -\rangle$ |

Encoding rule: start in $\lvert 0\rangle$, apply $X$ if the bit is 1, then apply $H$ if the basis is X.

In [4]:
def alice_encode(bit: int, basis: int) -> QuantumCircuit:
    """Build the qubit Alice sends for (bit, basis). No measurement."""
    qc = QuantumCircuit(1)
    if bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)
    return qc

# Show all four prep circuits as a sanity check.
state_names = {(0, 0): "|0>", (1, 0): "|1>", (0, 1): "|+>", (1, 1): "|->"}
for basis in (0, 1):
    for bit in (0, 1):
        print(f"  bit={bit}, basis={'Z' if basis == 0 else 'X'}  ->  {state_names[(bit, basis)]}")
        print(alice_encode(bit, basis).draw(fold=-1))

  bit=0, basis=Z  ->  |0>
   
q: 
   
  bit=1, basis=Z  ->  |1>
   ┌───┐
q: ┤ X ├
   └───┘
  bit=0, basis=X  ->  |+>
   ┌───┐
q: ┤ H ├
   └───┘
  bit=1, basis=X  ->  |->
   ┌───┐┌───┐
q: ┤ X ├┤ H ├
   └───┘└───┘


## Bob — measurement

If Bob's basis matches Alice's, he gets her bit. If it doesn't, his outcome is a fair coin flip.

To measure in the X basis, apply $H$ first, then measure in Z.

In [5]:
def bob_measure(received: QuantumCircuit, basis: int) -> int:
    """Measure the incoming qubit in Bob's chosen basis; return 0 or 1."""
    # Rebuild on a 1-qubit, 1-classical-bit circuit so measure(0,0) works regardless
    # of whether the incoming circuit had a classical register.
    qc = QuantumCircuit(1, 1)
    qc.compose(received, inplace=True)
    if basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    tqc = transpile(qc, backend)
    counts = backend.run(tqc, shots=1).result().get_counts()
    return int(next(iter(counts)))

## Run the protocol

In [6]:
# Protocol parameters
N         = 100    # number of qubits Alice sends
THRESHOLD = 0.15   # abort if observed error rate exceeds this

print("=" * 56)
print("  BB84 — no attacker")
print("=" * 56)
print(f"N = {N}, abort threshold = {THRESHOLD:.0%}\n")

  BB84 — no attacker
N = 100, abort threshold = 15%



In [7]:
# Step 1 — Alice picks her bits and bases
print("[ALICE] picking bits and bases")
alice_bits  = quantum_random_bits(N)
alice_bases = quantum_random_bits(N)

print(f"[ALICE]  bits  (first 20): {alice_bits[:20]}")
print(f"[ALICE]  bases (first 20): {alice_bases[:20]}   (0=Z, 1=X)")

[ALICE] picking bits and bases
[ALICE]  bits  (first 20): [1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1]
[ALICE]  bases (first 20): [0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0]   (0=Z, 1=X)


In [8]:
# Step 2 — Bob picks his measurement bases
print("[BOB] picking measurement bases")
bob_bases = quantum_random_bits(N)

print(f"[BOB]    bases (first 20): {bob_bases[:20]}   (0=Z, 1=X)")

[BOB] picking measurement bases
[BOB]    bases (first 20): [0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0]   (0=Z, 1=X)


In [9]:
# Step 3 — Alice transmits, Bob measures
print("[CHANNEL] Alice encodes -> qubit travels -> Bob measures")
bob_bits = []
for i in range(N):
    qc_sent  = alice_encode(alice_bits[i], alice_bases[i])
    bob_bits.append(bob_measure(qc_sent, bob_bases[i]))

print(f"[BOB]    measured (first 20): {bob_bits[:20]}")

[CHANNEL] Alice encodes -> qubit travels -> Bob measures
[BOB]    measured (first 20): [1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1]


In [10]:
# Step 4 — Sifting (over the classical channel)
print("[CLASSICAL] Alice and Bob announce bases, drop the mismatches")

keep = [i for i in range(N) if alice_bases[i] == bob_bases[i]]
sifted_a = [alice_bits[i] for i in keep]
sifted_b = [bob_bits[i]   for i in keep]

print(f"[SIFT]   kept {len(keep)} / {N} positions (expected ~{N//2})")
print(f"[ALICE]  sifted (first 20): {sifted_a[:20]}")
print(f"[BOB]    sifted (first 20): {sifted_b[:20]}")

[CLASSICAL] Alice and Bob announce bases, drop the mismatches
[SIFT]   kept 54 / 100 positions (expected ~50)
[ALICE]  sifted (first 20): [1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1]
[BOB]    sifted (first 20): [1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1]


In [11]:
# Step 5 — Error check
# Sacrifice the first ~30% of the sifted key as a public comparison sample.
# Those bits are revealed and discarded; the rest becomes the shared key.
print("[CLASSICAL] revealing a sample of the sifted key to estimate the error rate")

sample_n = max(1, len(sifted_a) // 3)
errs     = sum(1 for a, b in zip(sifted_a[:sample_n], sifted_b[:sample_n]) if a != b)
err_rate = errs / sample_n

print(f"\n[CHECK]  sample size : {sample_n}")
print(f"[CHECK]  mismatches  : {errs}")
print(f"[CHECK]  error rate  : {err_rate:.2%}")
print(f"[CHECK]  threshold   : {THRESHOLD:.2%}")

if err_rate > THRESHOLD:
    print("\n[ABORT] error rate above threshold — aborting key exchange")
else:
    key_a = sifted_a[sample_n:]
    key_b = sifted_b[sample_n:]
    print("\n[OK]    error rate within threshold — channel looks clean")
    print(f"[KEY]   final length : {len(key_a)} bits")
    print(f"[KEY]   alice (first 20): {key_a[:20]}")
    print(f"[KEY]   bob   (first 20): {key_b[:20]}")
    print(f"[KEY]   keys identical : {key_a == key_b}")

[CLASSICAL] revealing a sample of the sifted key to estimate the error rate

[CHECK]  sample size : 18
[CHECK]  mismatches  : 0
[CHECK]  error rate  : 0.00%
[CHECK]  threshold   : 15.00%

[OK]    error rate within threshold — channel looks clean
[KEY]   final length : 36 bits
[KEY]   alice (first 20): [0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0]
[KEY]   bob   (first 20): [0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0]
[KEY]   keys identical : True


## Summary

With no attacker on a noiseless channel:
- about half the qubits survive sifting (bases agree on ~50% of positions),
- at every sifted position Bob's measurement reproduces Alice's bit exactly, so the observed error rate is 0%,
- the unrevealed sifted bits form a shared secret key that Alice and Bob agree on bit-for-bit.

The threshold of 15% is the deciding line. Anything above it would cause the protocol to abort. A ~25% error rate — what an intercept-and-resend attacker would cause — is well above 15%, so the same threshold would catch eavesdropping reliably.